# ALS: сочетание factors=32 и alpha=5

Сравниваем лучший отдельный вариант из опыта 07 (factors=64, alpha=5)
с сочетанием factors=32, alpha=5 на seed 42/7/2026. Остальные параметры:
regularization=0,1, iterations=20, num_threads=4; score_coef=0, planned=0,5,
фильтр 5/5, без затухания. Цели и пользователи validation прежние.
Основной критерий — средний warm-validation NDCG@10. Test здесь не оценивается.
Фиксируем seed=42 для последующего переобучения заранее, без выбора лучшего seed.


In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import sys
import json
from pathlib import Path
from time import perf_counter
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent))
from app.training.datasets import load_anilist, filter_interactions
from app.training.evaluate import chronological_split, relevant_items, evaluate, ndcg_at_k
from app.training.weights import weighted_interactions, DEFAULT_STATUS_WEIGHTS
from app.recsys.popularity import PopularityRecommender
from app.recsys.als import ALSRecommender

pd.set_option("display.float_format", "{:,.6f}".format)
VALIDATION_START = pd.Timestamp("2026-09-02T00:34:23Z")
TEST_START = pd.Timestamp("2026-09-30T00:34:23Z")
SNAPSHOT_END = pd.Timestamp("2026-10-28T00:34:23Z")
SCORE_COEF = 0.0
BASE_STATUS_WEIGHTS = DEFAULT_STATUS_WEIGHTS.copy()
assert BASE_STATUS_WEIGHTS == dict(completed=1.0, watching=1.0, on_hold=0.5, planned=0.5, dropped=0.0)
SEEDS = [42, 7, 2026]
ALS_PARAMS = dict(factors=64, regularization=0.1, alpha=20, iterations=20, num_threads=4)
versions = {name: importlib.metadata.version(name) for name in ["implicit", "numpy", "scipy", "pandas"]}
print(versions)
inter = load_anilist()
assert inter.updated_at.max() == SNAPSHOT_END, "Снимок изменился: пересмотрите протокол"
history, validation, _ = chronological_split(
    inter, validation_start=VALIDATION_START, test_start=TEST_START)
training = filter_interactions(history, min_user_items=5, min_item_users=5)
counts = history.user_id.value_counts()
targets = {u: items for u, items in relevant_items(validation).items() if counts.get(u, 0) >= 5}
users = sorted(targets)
seen = history.groupby("user_id").anime_id.agg(set).to_dict()
print(f"Train после фильтра: {len(training):,} записей; warm-validation: {len(users):,} пользователей")
assert len(users) == 7184
assert len(history) == 3513393 and len(training) == 3499908
assert len(validation) == 218925
# Конфигурации фиксируем до просмотра результатов.
CONFIGS = {
    "factors64_alpha5": {**ALS_PARAMS, "alpha": 5},
    "factors32_alpha5": {**ALS_PARAMS, "factors": 32, "alpha": 5},
}
weighted = weighted_interactions(training, status_weights=BASE_STATUS_WEIGHTS.copy(),
                                 score_coef=SCORE_COEF, min_score=5, half_life_days=None)
assert len(weighted) == 3293589
assert not weighted.status.eq("dropped").any()
assert not weighted.score.lt(5).fillna(False).any()
assert DEFAULT_STATUS_WEIGHTS == BASE_STATUS_WEIGHTS
out = Path.cwd().parent / "artifacts" / "als_combined_parameters"
out.mkdir(parents=True, exist_ok=True)


/home/deonik3/AniFolder/ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'implicit': '0.7.3', 'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0'}


Train после фильтра: 3,499,908 записей; warm-validation: 7,184 пользователей


## Контрольная популярность

Прежний baseline за 28 дней на train 5/5: NDCG@10 ≈ 0,113021.

In [2]:
popular = PopularityRecommender(window_days=28).fit(training, now=VALIDATION_START, history=history)
popular_metrics = evaluate(popular.recommend_many(users, k=10), targets)
assert np.isclose(popular_metrics["ndcg@10"], 0.113021, atol=5e-7, rtol=0)
pd.Series(popular_metrics, name="популярность 28 дней").to_frame()

,популярность 28 дней
precision@10,0.099624
recall@10,0.064522
ndcg@10,0.113021
coverage@10,65.000000
users,"7,184.000000"


## Шесть запусков

Одинаковые положительные пары и полная история исключений во всех запусках.

In [3]:
records = []
per_user = {}
(out / "progress.log").write_text("")
for name, params in CONFIGS.items():
    for seed in SEEDS:
        started = perf_counter()
        model = ALSRecommender(**params, random_state=seed).fit(weighted, history=history)
        fit_seconds = perf_counter() - started
        started = perf_counter()
        recs = model.recommend_many(users, k=10)
        recommend_seconds = perf_counter() - started
        assert set(recs) == set(users)
        assert all(len(r) == len(set(r)) and not (set(r) & seen.get(u, set())) for u, r in recs.items())
        metrics = evaluate(recs, targets)
        per_user[(name, seed)] = np.array([ndcg_at_k(recs[u], targets[u], 10) for u in users])
        records.append({"configuration": name, "seed": seed, **params, **metrics,
                        "training_rows": len(weighted),
                        "short_lists": sum(len(r) < 10 for r in recs.values()),
                        "fit_seconds": fit_seconds, "recommend_seconds": recommend_seconds})
        message = (f"{len(records)}/{len(CONFIGS)*len(SEEDS)}: {name}, seed={seed}: "
                   f"NDCG@10={metrics['ndcg@10']:.6f}, coverage={metrics['coverage@10']}, "
                   f"train={fit_seconds:.1f}s")
        print(message, flush=True)
        with (out / "progress.log").open("a") as log:
            log.write(message + "\n")
results = pd.DataFrame(records)
display(results.set_index(["configuration", "seed"]))

1/6: factors64_alpha5, seed=42: NDCG@10=0.108484, coverage=3103, train=7.4s


2/6: factors64_alpha5, seed=7: NDCG@10=0.106986, coverage=3103, train=6.8s


3/6: factors64_alpha5, seed=2026: NDCG@10=0.107470, coverage=3109, train=7.1s


4/6: factors32_alpha5, seed=42: NDCG@10=0.107107, coverage=2827, train=6.1s


5/6: factors32_alpha5, seed=7: NDCG@10=0.107036, coverage=2825, train=5.8s


6/6: factors32_alpha5, seed=2026: NDCG@10=0.108338, coverage=2805, train=5.6s


factors  regularization  alpha  iterations  \
configuration    seed                                               
factors64_alpha5 42         64        0.100000      5          20   
                 7          64        0.100000      5          20   
                 2026       64        0.100000      5          20   
factors32_alpha5 42         32        0.100000      5          20   
                 7          32        0.100000      5          20   
                 2026       32        0.100000      5          20   

                       num_threads  precision@10  recall@10  ndcg@10  \
configuration    seed                                                  
factors64_alpha5 42              4      0.089797   0.066203 0.108484   
                 7               4      0.088906   0.065011 0.106986   
                 2026            4      0.088850   0.064417 0.107470   
factors32_alpha5 42              4      0.089087   0.063774 0.107107   
                 7               4      0.089101   0.063391 0.107036   
                 2026            4      0.090117   0.064640 0.108338   

                       coverage@10  users  training_rows  short_lists  \
configuration    seed                                                   
factors64_alpha5 42           3103   7184        3293589            0   
                 7            3103   7184        3293589            0   
                 2026         3109   7184        3293589            0   
factors32_alpha5 42           2827   7184        3293589            0   
                 7            2825   7184        3293589            0   
                 2026         2805   7184        3293589            0   

                       fit_seconds  recommend_seconds  
configuration    seed                                  
factors64_alpha5 42       7.435653           9.147456  
                 7        6.776838           9.112131  
                 2026     7.076859           9.373249  
factors32_alpha5 42       6.051229           8.952787  
                 7        5.752214           8.880908  
                 2026     5.602188           8.826012

## Качество и чувствительность к seed

Std описывает инициализацию на одном validation. Подбор среди нескольких вариантов может завышать качество выбранного кандидата; три seed не заменяют независимую проверку.

In [4]:
summary = results.groupby("configuration", sort=False).agg(
    ndcg_mean=("ndcg@10", "mean"), ndcg_std=("ndcg@10", "std"),
    ndcg_min=("ndcg@10", "min"), ndcg_max=("ndcg@10", "max"),
    precision_mean=("precision@10", "mean"), recall_mean=("recall@10", "mean"),
    coverage_mean=("coverage@10", "mean"), fit_seconds_mean=("fit_seconds", "mean"),
    recommend_seconds_mean=("recommend_seconds", "mean"))
display(summary.sort_values("ndcg_mean", ascending=False))
candidate = summary.ndcg_mean.idxmax()
print("Лучший вариант по среднему NDCG:", candidate, CONFIGS[candidate])
print("NDCG baseline популярности:", popular_metrics["ndcg@10"])
paired = results.pivot(index="seed", columns="configuration", values="ndcg@10")
paired_deltas = paired.subtract(paired["factors64_alpha5"], axis=0)
display(paired_deltas)
delta_rows = []
for name in CONFIGS:
    if name == "factors64_alpha5":
        continue
    delta = np.mean([per_user[(name, seed)] - per_user[("factors64_alpha5", seed)] for seed in SEEDS], axis=0)
    delta_rows.append({"configuration": name, "mean_delta": delta.mean(),
                       "лучше": int((delta > 1e-12).sum()),
                       "без изменения": int((np.abs(delta) <= 1e-12).sum()),
                       "хуже": int((delta < -1e-12).sum())})
user_deltas = pd.DataFrame(delta_rows).set_index("configuration")
display(user_deltas)
# Контроль воспроизведения опыта 07, включая coverage, precision и recall.
previous = pd.read_csv(out.parent / "als_hyperparameters" / "runs.csv")
metric_columns = ["ndcg@10", "precision@10", "recall@10", "coverage@10"]
old = previous[previous.configuration.eq("alpha_5")].set_index("seed").loc[SEEDS, metric_columns]
new = results[results.configuration.eq("factors64_alpha5")].set_index("seed").loc[SEEDS, metric_columns]
np.testing.assert_allclose(new, old, rtol=0, atol=1e-12)
assert results.short_lists.eq(0).all()
print("Контроль опыта 07 воспроизведён; полный топ-10 во всех шести запусках.")


,ndcg_mean,ndcg_std,ndcg_min,ndcg_max,precision_mean,recall_mean,coverage_mean,fit_seconds_mean,recommend_seconds_mean
configuration,,,,,,,,,
factors64_alpha5,0.107646,0.000765,0.106986,0.108484,0.089184,0.065210,"3,105.000000",7.096450,9.210945
factors32_alpha5,0.107494,0.000732,0.107036,0.108338,0.089435,0.063935,"2,819.000000",5.801877,8.886569


Лучший вариант по среднему NDCG: factors64_alpha5 {'factors': 64, 'regularization': 0.1, 'alpha': 5, 'iterations': 20, 'num_threads': 4}
NDCG baseline популярности: 0.11302101779723082


configuration,factors32_alpha5,factors64_alpha5
seed,,
7,0.000051,0.000000
42,-0.001377,0.000000
2026,0.000869,0.000000


,mean_delta,лучше,без изменения,хуже
configuration,,,,
factors32_alpha5,-0.000152,1662,3714,1808


Контроль опыта 07 воспроизведён; полный топ-10 во всех шести запусках.


## Артефакты опыта

Результаты в `ml/artifacts/als_combined_parameters/` (вне Git).

In [5]:
results.to_csv(out / "runs.csv", index=False)
summary.to_csv(out / "summary.csv")
paired_deltas.to_csv(out / "paired_deltas.csv")
user_deltas.to_csv(out / "user_deltas.csv")
(out / "protocol.json").write_text(json.dumps({
    "snapshot_end": SNAPSHOT_END.isoformat(), "validation_start": VALIDATION_START.isoformat(),
    "test_start": TEST_START.isoformat(), "test_evaluated": False,
    "min_user_items": 5, "min_item_users": 5, "score_coef": SCORE_COEF,
    "status_weights": BASE_STATUS_WEIGHTS, "min_score": 5, "min_user_scores": 5,
    "clip": [0.5, 1.5], "half_life_days": None, "seeds": SEEDS,
    "configurations": CONFIGS, "confidence": "1 + alpha * weight",
    "search_design": "combined factors and alpha comparison", "selection_metric": "mean warm-validation ndcg@10",
    "candidate": candidate, "final_seed": 42, "selected_parameters": CONFIGS[candidate], "warm_validation_users": len(users),
    "target_filtering": False, "exclusion_history": "full train",
    "user_id_namespace": "anilist", "item_id_namespace": "anilist",
    "versions": versions, "popular_baseline": popular_metrics,
}, ensure_ascii=False, indent=2) + "\n")
print("Сохранены:", ", ".join(p.name for p in sorted(out.iterdir())))

Сохранены: paired_deltas.csv, progress.log, protocol.json, runs.csv, summary.csv, user_deltas.csv


## Вывод

| Вариант | Средний NDCG@10 | Std | Recall@10 | Coverage@10 |
|---|---:|---:|---:|---:|
| factors=64, alpha=5 | 0.107646 | 0.000765 | 0.065210 | 3105 |
| factors=32, alpha=5 | 0.107494 | 0.000732 | 0.063935 | 2819 |

Сочетание не улучшило средний NDCG: разница −0,000152 (−0,14%).
Разницы относительно 64 факторов для seed 42/7/2026:
−0,001377 / +0,000051 / +0,000869. Устойчивого выигрыша по seed нет;
эта небольшая разница не доказывает статистическое превосходство 64 факторов.
У 32 факторов ниже средние recall и покрытие, обучение немного быстрее.

По заранее выбранному критерию сохраняем **factors=64, alpha=5,
regularization=0,1, iterations=20, num_threads=4**. Для финального обучения
заранее зафиксирован seed=42. Веса: score_coef=0, planned=0,5, фильтр 5/5,
оценки <5 исключены, затухание отсутствует.
Популярность за 28 дней остаётся выше обоих вариантов: NDCG@10=0,113021.

Контроль точно воспроизвёл метрики опыта 07 на всех seed. Все шесть запусков
вернули полный топ-10 для 7184 пользователей, без дублей и просмотренных тайтлов.
Test в этом опыте не оценивался. Следующий этап — ноутбук 09:
переобучение на train+validation и финальная оценка ранее выбранных алгоритмов.
